In [1]:
# stella embedding and pca
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model
model_name = "NovaSearch/stella_en_400M_v5"
tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)    
model = AutoModel.from_pretrained(model_name,trust_remote_code=True)


/home/yummareddy/anaconda3/envs/test/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at NovaSearch/stella_en_400M_v5 were not used when initializing NewModel: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [2]:
import torch
import numpy as np
from tqdm import tqdm

def encode_stella_batched(texts, tokenizer, model, batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu"):
    model = model.to(device)
    model.eval()
    
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            output = model(**encoded)
            # Mean pooling
            attention_mask = encoded['attention_mask'].unsqueeze(-1)
            last_hidden = output.last_hidden_state * attention_mask
            pooled = last_hidden.sum(1) / attention_mask.sum(1)
            embeddings.append(pooled.cpu())

    return torch.cat(embeddings, dim=0).numpy()


In [3]:
import pandas as pd
df = pd.read_csv('../api_fetcher/cleaned_abstracts.csv')
df_list = df['abstracts'].tolist()

In [4]:
df_list = df['abstracts'].tolist()

In [5]:
embeddings = encode_stella_batched(df_list, tokenizer, model)

Encoding:   0%|                                                                                                                                  | 0/211 [00:00<?, ?it/s]


NotImplementedError: No operator found for `memory_efficient_attention_forward` with inputs:
     query       : shape=(1, 3335, 16, 64) (torch.float32)
     key         : shape=(1, 3335, 16, 64) (torch.float32)
     value       : shape=(1, 3335, 16, 64) (torch.float32)
     attn_bias   : <class 'xformers.ops.fmha.attn_bias.BlockDiagonalMask'>
     p           : 0.0
`fa3F@2.7.4` is not supported because:
    device=cpu (supported: {'cuda'})
    dtype=torch.float32 (supported: {torch.bfloat16, torch.float16})
`fa2F@2.5.7-pt` is not supported because:
    device=cpu (supported: {'cuda'})
    dtype=torch.float32 (supported: {torch.bfloat16, torch.float16})
`cutlassF-pt` is not supported because:
    device=cpu (supported: {'cuda'})

In [ ]:
 # Returns a list of sentence embeddings

np.save("stella_encoded_data.npy", embeddings)

In [ ]:
# stella embedding and pca
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from tqdm import tqdm
import pandas as pd

# Load tokenizer and model
model_name = "NovaSearch/stella_en_400M_v5"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)    
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    attn_implementation="eager"  # This fixes the xformers issue
)

def encode_stella_batched(texts, tokenizer, model, batch_size=16, device=None):
    # Auto-detect device if not specified
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model = model.to(device)
    model.eval()
    
    embeddings = []
    
    # Process in batches
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i:i+batch_size]
        
        # Tokenize with proper truncation
        encoded = tokenizer(
            batch, 
            padding=True, 
            truncation=True, 
            return_tensors="pt",
            max_length=512  # Explicitly set max length
        ).to(device)
        
        with torch.no_grad():
            output = model(**encoded)
            
            # Mean pooling
            attention_mask = encoded['attention_mask'].unsqueeze(-1)
            last_hidden = output.last_hidden_state * attention_mask
            pooled = last_hidden.sum(1) / attention_mask.sum(1)
            embeddings.append(pooled.cpu())
    
    return torch.cat(embeddings, dim=0).numpy()

# Load and process data
df = pd.read_csv('../api_fetcher/cleaned_abstracts.csv')
df_list = df['abstracts'].tolist()

print(f"Encoding {len(df_list)} abstracts...")
embeddings = encode_stella_batched(df_list, tokenizer, model)

# Save embeddings
np.save("stella_encoded_data.npy", embeddings)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Saved to stella_encoded_data.npy")